# A1.8 · Unexpected code execution

**Function A — Securing AI Architectures → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.7 · Identity spoofing and impersonation](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**.

| | |
|---|---|
| Open-source tooling | Falco, gVisor |
| Open-weight models | GLM-4.6 |
| Frontier models | Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Asking a model to write code is safe. Running the code it wrote is the part that is not, and most agent frameworks ship the second one enabled with the same process privileges as the framework itself.

## 2 · The framework

```
   model ---> "here is a script that does it" ---> agent runtime
                                                        |
                                            exec() on the host
                                                        v
                                        whatever the PROCESS can reach:
                                        files . network . credentials . socket

   writing the code is safe. running it is the part that is not.
```

**OWASP T11 — Unexpected RCE and Code Attacks. LLM05 — Improper Output Handling.**

Many useful agents write code and run it — that is what makes a data-analysis
agent or a coding agent worth having. The **agent_runtime** component executes
text the **model** produced, on a host, in a process.

The risk is not exotic. Model-authored code is just code, and it runs with
whatever the process has: the filesystem it can see, the network it can reach,
and every credential in its environment. There is no privilege boundary between
"the code the agent wrote to reformat a CSV" and "the code that reads
`~/.aws/credentials`", because both are strings passed to the same interpreter.

Two paths lead here, and only one involves an attacker:

**Steered.** An injection from A1.3 tells the agent to write particular code.
The runtime executes it because executing code is its job.

**Unsteered.** Nobody attacked anything. The agent wrote something plausible and
wrong — a cleanup routine with a path variable that resolves higher than
intended — and the blast radius was decided by the environment, not by intent.

That second path is worth sitting with. Most teams model this as an attack. In
practice the first incident is usually an ordinary bug with production
credentials in scope, which is why the control in A3.2 is about what the process
can *reach*, not about what the model can be persuaded to *write*.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

What the executing process can reach, enumerated rather than assumed. Nothing below actually touches your machine — the environment is a fixture, so the lesson runs anywhere.

In [ ]:
# a stand-in for the process the agent's code runs inside
PROCESS_ENV = {
 "AWS_ACCESS_KEY_ID": "AKIA-EXAMPLE-NOT-REAL",
 "DATABASE_URL": "postgres://app:pw@prod-db/main",
 "HOME": "/home/agent",
}
FILESYSTEM = {"/home/agent/work/data.csv": "id,amount",
              "/home/agent/.ssh/id_ed25519": "PRIVATE KEY MATERIAL",
              "/etc/passwd": "root:x:0:0"}
NETWORK_REACHABLE = ["prod-db:5432", "169.254.169.254:80", "0.0.0.0/0"]

def execute(code):
    """The runtime runs model-authored text. Reach is decided by the process,
    not by the code's intent."""
    reached = []
    if "environ" in code:  reached += [f"env:{k}" for k in sorted(PROCESS_ENV)]
    if "open(" in code:    reached += [f"file:{p}" for p in sorted(FILESYSTEM)]
    if "connect" in code:  reached += [f"net:{h}" for h in NETWORK_REACHABLE]
    return reached

BENIGN = "rows = open('/home/agent/work/data.csv').read()"     # nobody attacked anything
STEERED = "import os; d=os.environ; connect('169.254.169.254')"

for label, code in (("ordinary bug / benign task", BENIGN),
                    ("steered by an injection", STEERED)):
    reach = execute(code)
    print(f"{label}:")
    print(f"   code   : {code[:58]}")
    print(f"   reached: {len(reach)} things")
    for r in reach[:6]:
        print(f"      {r}")
    print()

print("The benign task reached every file the process can see, including a")
print("private key it had no reason to touch. It was not attacked - the code")
print("used open(), and open() sees what the process sees.")
print()
print("Blast radius here is a property of the environment. A3.2 changes the")
print("environment; no amount of instruction changes it.")
assert any("id_ed25519" in r for r in execute(BENIGN))

## What you just proved

Model-authored code is executed against a fixture environment and the reach is enumerated: an ordinary, unattacked task touches every file the process can see including a private key, and steered code reaches the environment credentials and the cloud metadata address.

## Your turn

For one agent that executes code, list what is in its process environment right now. The credentials in that list are the blast radius of the next ordinary bug, not of the next attack.

---

**Next → [A1.9 · Agent communication poisoning](https://spbreed.github.io/cyber-commons/lessons/A1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*